# DATA 298B — Final Integrated Four-Model Benchmark

Runs Ministral, Phi-4, Qwen, and Gemma on the same 50 held-out cases using controlled transaction, graph, and community retrieval. This notebook does not train models.

In [ ]:
# Configuration — run this notebook from top to bottom.
REPO_URL = 'https://github.com/aryaMehta26/Graph-Based-Fraud-Intelligence-Platform.git'
REPO_BRANCH = 'feature/298b-backend'
HELDOUT_DATASET = '/content/drive/MyDrive/DATA298B/heldout/aml_heldout.jsonl'
ROOT = '/content/drive/MyDrive/DATA298B'
OUTPUT_ROOT = '/content/drive/MyDrive/DATA298B/final_benchmark_integrated'
MODELS = ['ministral', 'phi', 'qwen', 'gemma']
N_LEGITIMATE = 10
N_SUSPICIOUS = 40
MAX_TOKENS = 700
SEED = 42

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, subprocess

REPO_DIR = Path('/content/Graph-Based-Fraud-Intelligence-Platform')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Repository ready:', REPO_DIR)

In [ ]:
%pip install -q -r requirements.txt
%pip install -q accelerate bitsandbytes sentencepiece huggingface_hub pyyaml
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Colab GPU runtime before running this notebook.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

In [ ]:
from huggingface_hub import login
login(add_to_git_credential=False)

In [ ]:
# Select exactly one shared held-out case set for all four models.
import json
from collections import Counter

records = [json.loads(line) for line in Path(HELDOUT_DATASET).read_text().splitlines() if line.strip()]
legitimate = [r for r in records if r['ground_truth']['decision'] == 'LEGITIMATE'][:N_LEGITIMATE]
suspicious_pool = [r for r in records if r['ground_truth']['decision'] == 'SUSPICIOUS']
patterns = ['FAN_IN', 'FAN_OUT', 'CYCLE', 'STACK', 'SCATTER_GATHER', 'GATHER_SCATTER', 'BIPARTITE', 'RANDOM']
selected_suspicious, used = [], set()
for pattern in patterns:
    for record in [r for r in suspicious_pool if r['ground_truth'].get('pattern') == pattern][:5]:
        selected_suspicious.append(record)
        used.add(record['case_id'])
for record in suspicious_pool:
    if len(selected_suspicious) >= N_SUSPICIOUS:
        break
    if record['case_id'] not in used:
        selected_suspicious.append(record)
        used.add(record['case_id'])
selected_records = legitimate + selected_suspicious
if len(selected_records) != 50:
    raise ValueError(f'Expected 50 cases, selected {len(selected_records)}')
print('Cases:', len(selected_records))
print('Decisions:', Counter(r['ground_truth']['decision'] for r in selected_records))
print('Patterns:', Counter(r['ground_truth'].get('pattern') for r in selected_records))

In [ ]:
# Controlled retrieval helper. The same three retrievals are used for every model.
import re, time, uuid, gc
from dataclasses import asdict
from datetime import datetime, timezone
from src.aml.tools import execute_tool
from src.aml.schemas import InvestigationReport

def parse_json_response(text):
    match = re.search(r'\{.*\}', text, re.S)
    if not match:
        raise ValueError('No JSON object found in model response')
    return json.loads(match.group(0))

def investigate_with_retrieval(backend, case, model_name, trace_dir):
    started = time.perf_counter()
    context = case.agent_context()
    tool_calls, retrieved = [], []
    for tool_name in ['transaction_lookup', 'graph_lookup', 'community_lookup']:
        call = {'name': tool_name, 'arguments': {}, 'success': False}
        try:
            result = execute_tool(tool_name, context, {})
            call.update({'success': True, 'result': result})
            retrieved.append({'tool': tool_name, 'result': result})
        except Exception as exc:
            call['error'] = str(exc)
        tool_calls.append(call)

    prompt = {
        'case_context': context,
        'retrieved_tool_results': retrieved,
        'instruction': 'Return the final grounded JSON report now.'
    }
    system = ('You are an AML investigator. Return JSON only with exactly these keys: '
              'decision, pattern, risk_level, confidence, evidence, recommended_actions, summary. '
              'Use only facts in the case context and retrieved tool results. Do not invent facts. '
              'decision must be SUSPICIOUS or LEGITIMATE. pattern must be one of FAN_IN, FAN_OUT, '
              'CYCLE, STACK, SCATTER_GATHER, GATHER_SCATTER, BIPARTITE, RANDOM, EMERGING_UNKNOWN, NONE. '
              'risk_level must be LOW, MEDIUM, HIGH, or CRITICAL.')
    response = backend.generate([
        {'role': 'system', 'content': system},
        {'role': 'user', 'content': json.dumps(prompt, default=str)}
    ], max_tokens=MAX_TOKENS, temperature=0.0)
    raw = response.get('text', '')
    try:
        obj = parse_json_response(raw)
        report = InvestigationReport(decision=obj.get('decision', 'SUSPICIOUS'), pattern=obj.get('pattern', 'EMERGING_UNKNOWN'), risk_level=obj.get('risk_level', 'MEDIUM'), confidence=float(obj.get('confidence', 0.0)), evidence=obj.get('evidence', []), recommended_actions=obj.get('recommended_actions', []), summary=obj.get('summary', ''))
    except Exception:
        report = InvestigationReport(summary='Invalid model output')
    trace = {'trace_id': uuid.uuid4().hex, 'case_id': case.case_id, 'model_family': model_name, 'variant': 'integrated_controlled_retrieval', 'seed': SEED, 'tool_budget': 3, 'tool_calls': tool_calls, 'tool_budget_hit': False, 'context': context, 'ground_truth': case.ground_truth, 'report': asdict(report), 'raw_response': raw, 'latency_seconds': time.perf_counter() - started, 'prompt_tokens': response.get('prompt_tokens', 0), 'output_tokens': response.get('output_tokens', 0), 'timestamp': datetime.now(timezone.utc).isoformat()}
    trace_dir.mkdir(parents=True, exist_ok=True)
    (trace_dir / f"{trace['trace_id']}.json").write_text(json.dumps(trace, indent=2, default=str))
    return trace

In [ ]:
# Long-running final benchmark. It resumes completed cases.
import yaml
from src.aml.backend import TransformersBackend
from src.aml.schemas import InvestigationCase
from src.evaluation.benchmark_runner import run as evaluate_traces

registry = yaml.safe_load((REPO_DIR / 'configs/models/registry.yaml').read_text())
root = Path(ROOT)
output_root = Path(OUTPUT_ROOT)

for MODEL in MODELS:
    print(f'\n===== {MODEL.upper()} =====', flush=True)
    adapter_dir = root / 'adapters' / f'{MODEL}_full_grounded'
    model_output = output_root / f'agentic_{MODEL}'
    trace_dir = model_output / 'traces'
    trace_jsonl = model_output / 'agentic_traces.jsonl'
    model_output.mkdir(parents=True, exist_ok=True)
    spec = registry[MODEL]
    backend = TransformersBackend(spec['model_id'], revision=spec['revision'], adapter_path=str(adapter_dir), model_kwargs={'load_in_4bit': True, 'device_map': 'auto'})
    completed = {json.loads(line)['case_id'] for line in trace_jsonl.read_text().splitlines() if line.strip()} if trace_jsonl.exists() else set()
    with trace_jsonl.open('a') as output:
        for index, record in enumerate(selected_records, 1):
            if record['case_id'] in completed:
                print(f'{MODEL}: {index}/50 {record["case_id"]} already complete', flush=True)
                continue
            case = InvestigationCase(case_id=record['case_id'], context=record['context'], ground_truth=record['ground_truth'], split=record.get('split', 'test'))
            print(f'{MODEL}: {index}/50 {case.case_id}', flush=True)
            trace = investigate_with_retrieval(backend, case, MODEL, trace_dir)
            output.write(json.dumps(trace, default=str) + '\n')
            output.flush()
    metrics = evaluate_traces(trace_jsonl, model_output / 'metrics')
    print(json.dumps(metrics, indent=2, default=str), flush=True)
    del backend
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# Save one side-by-side comparison file.
import pandas as pd
rows = []
for MODEL in MODELS:
    path = Path(OUTPUT_ROOT) / f'agentic_{MODEL}' / 'metrics' / 'benchmark_metrics.json'
    metrics = json.loads(path.read_text())
    row = {'model': MODEL}
    for section, values in metrics.items():
        if isinstance(values, dict):
            for key, value in values.items():
                if isinstance(value, (int, float, str)) or value is None:
                    row[f'{section}_{key}'] = value
    rows.append(row)
comparison = pd.DataFrame(rows)
comparison.to_csv(Path(OUTPUT_ROOT) / 'model_comparison_integrated.csv', index=False)
(Path(OUTPUT_ROOT) / 'model_comparison_integrated.json').write_text(comparison.to_json(orient='records', indent=2))
display(comparison)
print('Saved:', Path(OUTPUT_ROOT) / 'model_comparison_integrated.csv')

In [ ]:
# LAST CELL — run only after the comparison files are saved.
import gc, torch
try:
    del backend
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Integrated four-model benchmark complete. Disconnecting runtime...', flush=True)
from google.colab import runtime
runtime.unassign()